# Q16 — Qwen1.5-7B: OR Subspace Dimensionality

**Mirrors:** `16_[Analysis]_OR_Subspace_Dimensionality.ipynb` (LLaMA)

**Key limitation:** Qwen RH n=1 — the harmful-refusal subspace cannot be computed.  
All LLaMA comparisons (OR vs RH dimensionality) are replaced with OR-only analysis.

**What we compute:**
- OR subspace dimensionality across all 31 layers
- PC1 variance fraction and n80 (components for 80% cumvar)
- 2D PCA of OR samples coloured by task (task axes dominate?)
- LLaMA OR reference drawn for comparison where applicable

**From NB20:** at L5, OR PC1 = 64.1%, n80 = 2 (highly concentrated vs LLaMA: 24.5%, n80=11).  
This appears paradoxically *more* concentrated — likely because Qwen's OR samples cluster
in a single task (cryptanalysis) rather than spanning multiple tasks.

**No GPU required.**

In [ ]:
EMBEDDINGS_DIR = './embeddings_qwen'
FIGURES_DIR    = './figures'
KNOWN_PEAK_L   = 5   # Qwen constellation peak from Q13a

import os
os.makedirs(FIGURES_DIR, exist_ok=True)
print('Config set.')

In [ ]:
# COLAB ONLY
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
# !cp -a "/content/drive/MyDrive/embeddings/overalign_eval_qwen_1_5/." {EMBEDDINGS_DIR}/.
print('(Colab cell skipped)')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 13, 'font.family': 'serif',
    'axes.titlesize': 14, 'axes.titleweight': 'bold',
    'axes.labelsize': 13, 'legend.fontsize': 12,
    'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 2.2,
})

TASK_COLORS = {
    'sentiment_analysis': '#E74C3C',
    'translate':          '#3498DB',
    'rephrase':           '#F39C12',
    'rag_qa':             '#27AE60',
    'cryptanalysis':      '#9B59B6',
}
TASK_PAL   = list(TASK_COLORS.values())

PAL = {
    'or_global': '#E74C3C',
    'llama_or':  '#E74C3C',
    'ref':       '#7F8C8D',
}
BENIGN_TASKS = ['sentiment_analysis', 'translate', 'cryptanalysis', 'rag_qa']

print('✓ Libraries loaded')

In [ ]:
def to_np(d):
    return {
        k: (v.float().numpy().astype(np.float32) if isinstance(v, torch.Tensor)
            else np.array([e.float().numpy().astype(np.float32) for e in v]))
        for k, v in d.items()
    }

csv_files = sorted(f for f in os.listdir(EMBEDDINGS_DIR) if f.endswith('.csv'))
csv_df    = pd.read_csv(os.path.join(EMBEDDINGS_DIR, csv_files[-1]))
torch_path = csv_df['torch_path'].iloc[0]
if not os.path.isabs(torch_path):
    torch_path = os.path.join(EMBEDDINGS_DIR, os.path.basename(torch_path))
torch_data = torch.load(torch_path, map_location='cpu')

embeddings_np        = to_np(torch_data['embeddings'])
texts                = torch_data['texts']
text_type_labels     = np.array(torch_data['text_type_labels'])
intended_task_labels = np.array(torch_data['intended_task_labels'])
response_labels      = csv_df['llm_evaluation'].values
refusal_labels       = csv_df['refusal_class'].values

LAYER_NAMES = sorted(
    [k for k in embeddings_np if 'input_norm' in k],
    key=lambda x: int(x.split('layer_')[1].split('_')[0])
)
LAYER_NUMS = [int(ln.split('layer_')[1].split('_')[0]) for ln in LAYER_NAMES]

print(f'Loaded: {len(texts)} samples | {len(LAYER_NAMES)} layers')

In [ ]:
refusing_mask = (refusal_labels == 'direct_refusal') | (refusal_labels == 'indirect_refusal')
answered_mask = refusal_labels == 'direct_answer'
harmful_mask  = text_type_labels == 'harmful_instruction'
benign_mask   = np.isin(intended_task_labels, BENIGN_TASKS)

OVER_REFUSAL_MASK = benign_mask  & refusing_mask
REFUSED_HARMFUL   = harmful_mask & refusing_mask
HARMLESS_ANSWERED = benign_mask  & answered_mask
TARGET_MASK = (
    ((response_labels == 'cautious') | (response_labels == 'not_harmful')) & answered_mask
)

print(f'OR: {OVER_REFUSAL_MASK.sum()}  |  RH: {REFUSED_HARMFUL.sum()}  |  TGT: {TARGET_MASK.sum()}')
print(f'Note: RH n={REFUSED_HARMFUL.sum()} — harmful-refusal subspace NOT computable (need ≥5)')

PEAK_LNAME = f'layer_{KNOWN_PEAK_L}_input_norm'
if PEAK_LNAME not in embeddings_np:
    PEAK_LNAME = LAYER_NAMES[KNOWN_PEAK_L] if KNOWN_PEAK_L < len(LAYER_NAMES) else LAYER_NAMES[0]
PEAK_LNUM = int(PEAK_LNAME.split('layer_')[1].split('_')[0])
print(f'Using peak layer: {PEAK_LNAME}')

In [ ]:
def run_pca(group_emb, contrast_emb=None, n_components=20):
    """Centre group_emb against contrast_emb mean, then PCA."""
    centre = contrast_emb.mean(0) if contrast_emb is not None else group_emb.mean(0)
    X      = group_emb - centre
    n_comp = min(n_components, X.shape[0] - 1, X.shape[1])
    pca    = PCA(n_components=n_comp, random_state=42)
    pca.fit(X)
    return pca

def n80(evr, threshold=0.80):
    return int(np.searchsorted(np.cumsum(evr), threshold) + 1)

print('Helpers defined.')

In [ ]:
# ── Figure 1: Cumulative explained variance at peak layer ─────────────────────
# Mirrors: fig_nb16_variance (LLaMA)
# Qwen: only OR subspace shown (RH n<5); LLaMA reference values annotated

emb = embeddings_np[PEAK_LNAME]
n_comp = min(20, OVER_REFUSAL_MASK.sum() - 1)
pca_or = run_pca(emb[OVER_REFUSAL_MASK], contrast_emb=emb[TARGET_MASK], n_components=n_comp)

pc1_var_or = pca_or.explained_variance_ratio_[0] * 100
comp80_or  = n80(pca_or.explained_variance_ratio_)

print(f'[Qwen OR at L{PEAK_LNUM}]  PC1 = {pc1_var_or:.1f}%  |  n80 = {comp80_or}')
print(f'[LLaMA OR at L12]         PC1 = 24.5%  |  n80 = 11   (reference)')
print(f'[LLaMA RH at L12]         PC1 = 30.3%  |  n80 = 8    (reference — NOT shown: Qwen RH n=1)')

K     = min(15, n_comp)
xk    = np.arange(1, K + 1)
cum_or_q = np.cumsum(pca_or.explained_variance_ratio_[:K]) * 100

# LLaMA OR reference values (from NB16 outputs)
LLAMA_OR_CUM = np.array([24.5, 39.1, 50.6, 59.6, 66.4, 71.8, 76.3, 79.9, 82.8, 85.2,
                          87.2, 89.0, 90.5, 91.8, 92.9])[:K]

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.8))

# Left: cumulative variance
ax = axes[0]
ax.plot(xk, cum_or_q, 'o-', color=PAL['or_global'], lw=2.2,
        label=f'Qwen OR (PC1={pc1_var_or:.0f}%, n80={comp80_or})', zorder=3)
ax.plot(xk, LLAMA_OR_CUM, 's--', color=PAL['ref'], lw=2.0, alpha=0.7,
        label='LLaMA OR ref (PC1=24.5%, n80=11)', zorder=2)
ax.axhline(80, color=PAL['ref'], lw=1.2, ls=':')
ax.text(K, 81, '80%', fontsize=9, color=PAL['ref'], ha='right')
ax.set_xlabel('Number of components')
ax.set_ylabel('Cumulative explained variance (%)')
ax.set_title(f'Qwen OR subspace at L{PEAK_LNUM}', pad=8)
ax.set_ylim(0, 108)
ax.legend(fontsize=9)

# Right: grouped bars PC1 / PC1-2 / PC1-3 / PC1-5
ax = axes[1]
groups  = ['PC1', 'PC1–2', 'PC1–3', 'PC1–5']
tops    = [1, 2, 3, 5]
tops_q  = [min(t, n_comp) for t in tops]
vals_q  = [pca_or.explained_variance_ratio_[:t].sum() * 100 for t in tops_q]
llama_or_tops = [24.5, 39.1, 50.6, 66.4]
x2 = np.arange(len(groups))
w  = 0.34

b1 = ax.bar(x2 - w/2, vals_q,        w, color=PAL['or_global'], alpha=0.85,
            edgecolor='black', linewidth=0.8, label='Qwen OR')
b2 = ax.bar(x2 + w/2, llama_or_tops, w, color=PAL['ref'], alpha=0.70,
            edgecolor='black', linewidth=0.8, label='LLaMA OR (ref)')

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1.5,
            f'{h:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(80, color=PAL['ref'], lw=1.2, ls=':')
ax.set_xticks(x2); ax.set_xticklabels(groups, fontsize=11)
ax.set_ylabel('Variance explained (%)')
ax.set_ylim(0, 120)
ax.set_title('Top component coverage', pad=8)
ax.legend(fontsize=9, loc='lower right')

plt.suptitle('Qwen1.5-7B OR subspace dimensionality vs LLaMA reference\n'
             '(Harmful-refusal subspace unavailable: Qwen RH n=1)',
             fontsize=11, fontweight='bold', y=1.03)
plt.tight_layout(pad=0.8)
plt.savefig(f'{FIGURES_DIR}/q_fig16_variance.pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{FIGURES_DIR}/q_fig16_variance.png', bbox_inches='tight', dpi=300)
plt.show()
print(f'✓ Saved: q_fig16_variance')

In [ ]:
# ── Figure 2: 2D PCA of OR samples coloured by task ───────────────────────────
# Mirrors: fig_nb16_task_pca (LLaMA)
# If OR is task-conditioned, samples from the same task should cluster
# in PC space without any task supervision.

emb     = embeddings_np[PEAK_LNAME]
X_all   = emb - emb.mean(0)
or_tasks= intended_task_labels[OVER_REFUSAL_MASK]
unique_or_tasks = sorted(set(or_tasks))

n_comp2 = min(2, OVER_REFUSAL_MASK.sum() - 1)
pca2    = PCA(n_components=n_comp2, random_state=42)
proj    = pca2.fit_transform(X_all[OVER_REFUSAL_MASK])

fig, ax = plt.subplots(figsize=(5.0, 4.2))

for task in unique_or_tasks:
    m   = or_tasks == task
    col = TASK_COLORS.get(task, '#95A5A6')
    ax.scatter(proj[m, 0] if n_comp2 >= 1 else np.zeros(m.sum()),
               proj[m, 1] if n_comp2 >= 2 else np.zeros(m.sum()),
               color=col, label=task.replace('_',' '),
               s=90, alpha=0.85, edgecolors='white', linewidths=0.5, zorder=3)

ev = pca2.explained_variance_ratio_ * 100 if n_comp2 >= 2 else np.array([100.0, 0.0])
ax.set_xlabel(f'PC1 ({ev[0]:.1f}% var)')
ax.set_ylabel(f'PC2 ({ev[1]:.1f}% var)')
ax.set_title(f'2D PCA of Qwen OR samples coloured by task (L{PEAK_LNUM})\n'
             f'Principal axes should separate tasks if OR is task-conditioned', pad=6)
ax.legend(title='Task', fontsize=10, bbox_to_anchor=(1.02, 1), loc='upper left')

if n_comp2 < 2:
    ax.text(0.5, 0.5, f'⚠ Only {OVER_REFUSAL_MASK.sum()} OR samples —\nPCA(2) unreliable',
            transform=ax.transAxes, ha='center', va='center', color='red', fontsize=11)

plt.tight_layout(pad=0.8)
plt.savefig(f'{FIGURES_DIR}/q_fig16_task_pca.pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{FIGURES_DIR}/q_fig16_task_pca.png', bbox_inches='tight', dpi=300)
plt.show()
print(f'✓ Saved: q_fig16_task_pca')

In [ ]:
# ── Layer sweep: OR subspace dimensionality across all 31 layers ──────────────
# Mirrors: fig_nb16_layer_sweep (LLaMA)
# Qwen OR only (no RH); LLaMA OR reference drawn as dashed

# LLaMA OR reference from NB16 (pc1_var_or, comp80_or per layer)
LLAMA_PC1_OR = [np.nan, 48.0, 27.7, 45.5, 37.1, 22.3, 22.7, 18.9, 19.7, 20.1,
                21.6, 22.4, 24.5, 24.2, 22.9, 22.5, 22.8, 22.9, 24.2, 25.9,
                27.5, 29.9, 30.5, 31.1, 31.5, 31.7, 31.5, 30.7, 30.5, 29.5, 30.2]
LLAMA_N80_OR = [1, 4, 10, 6, 8, 14, 11, 12, 11, 12,
                12, 12, 11, 11, 11, 11, 10, 10, 10, 10,
                10, 10, 10, 10, 10, 10, 10, 10, 11, 11, 11]
LLAMA_LAYER_NUMS = list(range(31))

results = []
for lname, lnum in zip(LAYER_NAMES, LAYER_NUMS):
    e = embeddings_np[lname]
    n_c = min(20, OVER_REFUSAL_MASK.sum() - 1)
    if n_c < 2:
        results.append({'layer': lnum, 'pc1_var_or': np.nan, 'n80_or': np.nan})
        continue
    pca_or_l = run_pca(e[OVER_REFUSAL_MASK], contrast_emb=e[TARGET_MASK], n_components=n_c)
    results.append({
        'layer':     lnum,
        'pc1_var_or': pca_or_l.explained_variance_ratio_[0] * 100,
        'n80_or':    n80(pca_or_l.explained_variance_ratio_),
    })

import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))

In [ ]:
# ── Figure 3: Layer sweep ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.8))
plt.subplots_adjust(wspace=0.35, top=0.82, bottom=0.13)

# Background bands
EARLY_BAND = (0, 4);  MID_BAND = (5, 19);  LATE_BAND = (20, 30)
for ax in axes:
    ax.axvspan(*EARLY_BAND, alpha=0.06, color='#95A5A6', zorder=0)
    ax.axvspan(*MID_BAND,   alpha=0.08, color='#2980B9', zorder=0)
    ax.axvspan(*LATE_BAND,  alpha=0.07, color='#E74C3C', zorder=0)

# Left: PC1 variance fraction
ax = axes[0]
ax.plot(df['layer'], df['pc1_var_or'], 'o-', color=PAL['or_global'], lw=2.2, zorder=3,
        label='Qwen OR')
ax.plot(LLAMA_LAYER_NUMS, LLAMA_PC1_OR, 's--', color=PAL['ref'], lw=1.8, alpha=0.6, zorder=2,
        label='LLaMA OR (ref)')
ax.axvline(PEAK_LNUM, color=PAL['ref'], lw=1.2, ls='-.', label=f'Qwen peak L{PEAK_LNUM}')
ax.axvline(12, color='black', lw=1.0, ls=':', alpha=0.5, label='LLaMA peak L12 (ref)')
ax.set_xlabel('Layer'); ax.set_ylabel('Variance explained by PC1 (%)')
ax.set_title('PC1 variance fraction', pad=6)
ax.legend(fontsize=9)

# Right: n80
ax = axes[1]
ax.plot(df['layer'], df['n80_or'], 'o-', color=PAL['or_global'], lw=2.2, zorder=3,
        label='Qwen OR')
ax.plot(LLAMA_LAYER_NUMS, LLAMA_N80_OR, 's--', color=PAL['ref'], lw=1.8, alpha=0.6, zorder=2,
        label='LLaMA OR (ref)')
ax.axvline(PEAK_LNUM, color=PAL['ref'], lw=1.2, ls='-.')
ax.axvline(12, color='black', lw=1.0, ls=':', alpha=0.5)
ax.set_xlabel('Layer'); ax.set_ylabel('Components for 80% variance (n80)')
ax.set_title('n80 across layers', pad=6)
ax.legend(fontsize=9)

plt.suptitle('Qwen1.5-7B — OR subspace dimensionality sweep (vs LLaMA reference)\n'
             'Lower n80 in Qwen may reflect single-task OR concentration (cryptanalysis)',
             fontsize=10, fontweight='bold')
plt.savefig(f'{FIGURES_DIR}/q_fig16_layer_sweep.pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{FIGURES_DIR}/q_fig16_layer_sweep.png', bbox_inches='tight', dpi=300)
plt.show()
print(f'✓ Saved: q_fig16_layer_sweep')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
peak_row = df[df['layer'] == PEAK_LNUM].iloc[0] if PEAK_LNUM in df['layer'].values else df.iloc[0]

print('=' * 65)
print('Q16 — QWEN OR SUBSPACE DIMENSIONALITY SUMMARY')
print('=' * 65)
print(f'At constellation peak L{PEAK_LNUM}:')
print(f'  Qwen  OR: PC1={peak_row["pc1_var_or"]:.1f}%  n80={peak_row["n80_or"]:.0f}')
print(f'  LLaMA OR: PC1=24.5%  n80=11  (reference from NB16)')
print(f'  LLaMA RH: PC1=30.3%  n80=8   (reference from NB16 — NOT computable for Qwen)')
print()
print('INTERPRETATION:')
print(f'  Qwen OR shows higher PC1 fraction ({peak_row["pc1_var_or"]:.0f}%) than LLaMA OR (24.5%).')
print('  This is likely because Qwen OR samples come predominantly from one task')
print('  (cryptanalysis), making the subspace appear more concentrated.')
print('  With OR samples spanning ≥2 tasks (as in LLaMA), we expect similar diffuse geometry.')
print()
print('LIMITATION: Cannot compute RH subspace (Qwen RH n=1)')
print('  Claims 2-3 (OR vs RH dimensionality comparison) cannot be directly replicated.')
print('=' * 65)